# MRS analysis notebook

This notebook is a lightweight analysis driver for the repository.

It assumes the Python package in `src/mrs_pipeline` has been installed in editable mode:

```bash
pip install -e .
```

The notebook keeps the workflow explicit and reproducible:
1. configure paths
2. convert the raw scanner export to BIDS
3. run the preprocessing functions
4. run segmentation and analysis
5. combine summary tables and QC outputs


## Environment check

Load the same environment used for the Bash pipeline before running the notebook.  
The notebook does not perform installation steps itself.


In [ ]:
import shutil
import warnings
from pathlib import Path

import pandas as pd

import mrs_pipeline as mp

warnings.filterwarnings("ignore")

# Optional: make notebook output a bit more compact
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

print("fsl_anat:", shutil.which("fsl_anat"))
print("fsl_mrs:", shutil.which("fsl_mrs"))
print("spec2nii:", shutil.which("spec2nii"))
print("mrs_tools:", shutil.which("mrs_tools"))


## Configuration

Edit the paths below for your machine or cluster.  
Keep them outside the code if you prefer to load them from a small config file.


In [ ]:
# Raw scanner export (sub-XX folders with F3T_2022_021_NNN and 2022_021_NNN subfolders).
RAW_DIR = Path("/path/to/your/raw")

# BIDS dataset written by the conversion; also the data root of the pipeline
# (outputs go to DATA_DIR/derivatives/mrs).
DATA_DIR = Path("/path/to/your/bids")

# Basis set root used by the final analysis step.
BASIS_SETS_DIR = Path("/path/to/your/basis_sets")

# Subject filters.
# Use include to process only a subset, or leave it empty to process all subjects.
INCLUDE = []
EXCLUDE = ["sub-01", "sub-02"]

# Optional: set to True to favour edit-off outputs where relevant.
OFF = False

RAW_DIR.exists(), DATA_DIR.exists(), BASIS_SETS_DIR.exists()


## Run preprocessing

These steps prepare the derivatives used later in the analysis stage.

Uncomment the lines you need for a given run.


In [ ]:
# Convert the raw export to BIDS (anatomy + MRS, spec2nii for the twix files).
# Dry run first: check the plan and warnings, then set apply=True.
mp.convert_raw_to_bids(RAW_DIR, DATA_DIR, include=INCLUDE, exclude=EXCLUDE, apply=False)

# Quick-look visualisation of the converted MRS data
# mp.glance_from_derivatives(DATA_DIR, include=INCLUDE, exclude=EXCLUDE)

# Run the edited MRS preprocessing wrapper
mp.preproc(DATA_DIR, include=INCLUDE, exclude=EXCLUDE)

# Run anatomical preprocessing
mp.MRS_anat_process(DATA_DIR, include=INCLUDE, exclude=EXCLUDE)

# Run SVS segmentation
mp.svs_segment_process(DATA_DIR, include=INCLUDE, exclude=EXCLUDE)


## Run final MRS analysis

This step runs `fsl_mrs` using the basis sets and preprocessing outputs produced above.


In [ ]:
mp.mrs_analysis_process(
    DATA_DIR,
    BASIS_SETS_DIR,
    include=INCLUDE,
    exclude=EXCLUDE,
    off=OFF,
)


## Combine outputs

These functions collect subject-level outputs into summary tables.


In [ ]:
summary_df = mp.process_summary_files(DATA_DIR, off=OFF)
qc_df = mp.process_qc_files(DATA_DIR, off=OFF)
tissue_df = mp.process_tissue_fraction_files(DATA_DIR)

summary_df.head() if summary_df is not None else None


## Optional edit-off summaries

Only run these if you also want the alternative edit-off aggregation.


In [ ]:
# Edit-off tables
# mp.process_summary_files(DATA_DIR, off=True)
# mp.process_qc_files(DATA_DIR, off=True)
